## Fase 1: ETL - Unificación y Limpieza de los Registros de Accesos Diarios
En esta sección se consolida la información de accesos al gimnasio que se encontraba distribuida por días de la semana. El proceso automatizado busca y lee los múltiples archivos Excel diarios, los une en un único registro histórico omitiendo los índices originales y exporta el conjunto unificado a la carpeta de datos procesados para facilitar su manipulación posterior.

In [1]:
import pandas as pd
import glob

# 1. Definimos la ruta para localizar todos los archivos Excel con las entradas de acceso por días
ruta_busqueda = "../../datos/acceso_por_dias/acceso_*.xlsx"

archivos_excel = glob.glob(ruta_busqueda) 

# Comprobacion de seguridad para verificar la lectura de los archivos
print(f"Archivos encontrados por glob: {len(archivos_excel)}")
for archivo in archivos_excel:
    print(f" - Encontrado: {archivo}")

# Procedemos con la extraccion y union solo si se han detectado archivos
if len(archivos_excel) > 0:
    # 2. Leemos secuencialmente cada archivo y los almacenamos en una lista temporal
    lista_dataframes = [pd.read_excel(f) for f in archivos_excel]

    # 3. Concatenamos todos los DataFrames verticalmente en una unica tabla maestra
    df_accesos_total = pd.concat(lista_dataframes, ignore_index=True)
    
    # 3.1 Filtramos los datos para aislar unicamente a los socios con cuota mensual
    df_accesos_total = df_accesos_total[df_accesos_total['tipo_cuota'] == '1 (MENSUAL)']

    # 3.2 Eliminamos columnas con informacion personal que no aportan valor predictivo (PII)
    columnas_a_borrar = ['nombre', 'apellido'] 
    df_accesos_total = df_accesos_total.drop(columns=columnas_a_borrar, errors='ignore')

    # 4. Verificamos el volumen final tras la unificacion y limpieza
    print(f"\nExito! Total de registros limpios cargados: {len(df_accesos_total)}")
    
else:
    print("\nERROR: No se ha encontrado ningun archivo. Revisa la ruta.")

Archivos encontrados por glob: 7
 - Encontrado: ../../datos/acceso_por_dias\acceso_1_lunes.xlsx
 - Encontrado: ../../datos/acceso_por_dias\acceso_2_martes.xlsx
 - Encontrado: ../../datos/acceso_por_dias\acceso_3_miercoles.xlsx
 - Encontrado: ../../datos/acceso_por_dias\acceso_4_jueves.xlsx
 - Encontrado: ../../datos/acceso_por_dias\acceso_5_viernes.xlsx
 - Encontrado: ../../datos/acceso_por_dias\acceso_6_sabado.xlsx
 - Encontrado: ../../datos/acceso_por_dias\acceso_7_domingo.xlsx

Exito! Total de registros limpios cargados: 659042


## Exportación Final del Histórico de Accesos

In [2]:
import os

# 1. Definimos el directorio destino para almacenar los datos consolidados
ruta_guardado = "../../datos/procesados"

# 2. Creamos la carpeta de destino en caso de que no exista en el sistema
os.makedirs(ruta_guardado, exist_ok=True)

# 3. Exportamos el DataFrame unificado a un archivo CSV omitiendo el indice original
ruta_archivo_final = f"{ruta_guardado}/accesos_historico_total.csv"
df_accesos_total.to_csv(ruta_archivo_final, index=False)

# 4. Imprimimos un mensaje de confirmacion indicando la ubicacion del archivo
print(f"Archivo guardado correctamente:\n{ruta_archivo_final}")

Archivo guardado correctamente:
../../datos/procesados/accesos_historico_total.csv
